# 🛡️ PhishGuard 2.0: Consolidated Master Dataset Training Engine
### Accelerated for Google Colab NVIDIA Tesla T4 GPU (16 GB VRAM)

This notebook consolidates the three source corpora into a single deduplicated master
dataset, trains all production models on it, evaluates them on a strict domain-grouped
15% held-out test set, and packages the artifacts into `saved_models.zip`.

**Pipeline Components:**
- 🗄️ **Master Dataset Consolidation** (3 corpora → one normalised, deduplicated, group-tagged table)
- ⚡ **30-Dimension Heuristic Signal Extractor** (length-pruned brand matching, pre-compiled regex)
- 🧠 **PyTorch Deep Residual PhishNet** (`PhishNetDeep`, FP16 Tensor Core mixed precision)
- 🌲 **LightGBM Boosted Tree Ensembles** (GPU/CUDA with high-speed CPU fallback)
- 📝 **Character Subword NLP Classifier** (TF-IDF 20k features + calibrated SGD)
- 🎯 **Dynamic Mixture-of-Experts Gating** + FPR-constrained threshold tuning

**Upload either** the three raw CSVs, **or** just `phishguard_master_dataset.parquet`
(~45 MB) — the trainer prefers the master file and falls back to the raw CSVs.


## 1. Hardware & GPU Diagnostic Check

In [ ]:
# Verify GPU assignment and CUDA capabilities
!nvidia-smi

import torch
print("=" * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    cap = torch.cuda.get_device_capability(0)
    print(f"✅ GPU Active: {gpu_name}")
    print(f"✅ GPU Memory: {vram:.2f} GB GDDR6 VRAM")
    print(f"✅ Compute Capability: {cap[0]}.{cap[1]} (FP16 Tensor Cores Supported)")
    print(f"✅ PyTorch CUDA: {torch.version.cuda}")
else:
    print("⚠️ GPU NOT DETECTED!")
    print("To enable T4 GPU:")
    print("  1. Go to menu: Runtime -> Change runtime type")
    print("  2. Set Hardware accelerator to 'T4 GPU'")
    print("  3. Click 'Save' and re-run this cell.")
print("=" * 60)

## 2. Install High-Performance Dependencies

In [ ]:
# 1. Configure OpenCL / GPU acceleration drivers for Colab T4 GPU
!apt-get install -y -qq ocl-icd-libopencl1 opencl-headers clinfo libboost-all-dev > /dev/null 2>&1
!mkdir -p /etc/OpenCL/vendors && echo "libnvidia-opencl.so.1" > /etc/OpenCL/vendors/nvidia.icd 2>/dev/null || true

# 2. Install optimized training dependencies.
#    pyarrow is required to read/write phishguard_master_dataset.parquet.
!pip install torch torchvision lightgbm scikit-learn pandas numpy scipy joblib tqdm pyarrow matplotlib seaborn -q
print("✅ Dependencies & GPU acceleration drivers configured!")

## 3. Consolidate the Master Dataset

Merges `malicious_phish.csv`, `phishing_site_urls.csv` and
`PhiUSIIL_Phishing_URL_Dataset.csv` into one normalised, cross-corpus deduplicated,
consensus-labelled and domain-grouped table.

Skipped automatically if you uploaded `phishguard_master_dataset.parquet` directly.

**Read the surface-form audit it prints.** `path_rule_accuracy` is the accuracy of
predicting "phishing iff the URL has a path". Near 0.50 means URL shape carries no class
signal, which is what you want. Near 1.00 means the corpus is separable on formatting
alone and a model trained on it will score brilliantly here and fail in production —
PhiUSIIL alone sits at 0.6997 for exactly this reason; diluted into the master it drops
to 0.5264.

`--conflict-policy` decides what happens to the ~90k URLs the two legacy corpora label
differently. `phishing` (default) sends every tie to phishing; `drop` removes them for a
smaller but cleaner table.

In [ ]:
import os

if os.path.exists('phishguard_master_dataset.parquet') or os.path.exists('phishguard_master_dataset.csv'):
    print("✅ Master dataset already present — skipping consolidation.")
else:
    !python consolidate_datasets.py            # add --conflict-policy drop for the cleaner table

import json
if os.path.exists('master_dataset_manifest.json'):
    mm = json.load(open('master_dataset_manifest.json'))
    t = mm['totals']
    print(f"
📊 {t['unique_urls']:,} URLs | {t['benign']:,} benign / {t['phishing']:,} phishing")
    print(f"   {t['unique_registered_domains']:,} registered domains ({t['urls_per_domain']} URLs each)")
    print(f"   path-rule accuracy: {mm['surface_form_audit']['master']['path_rule_accuracy']} (0.50 = no shortcut)")
    print(f"   label conflicts: {t['label_conflicts']:,} (policy: {mm['conflict_policy']})")

## 4. Run the Full Production Training Pipeline

Executes master-dataset ingestion, 30-D feature extraction, TF-IDF vectorization,
PyTorch PhishNet training, LightGBM tree training, MoE threshold optimization and
artifact serialization.

Use `--quick-test` first to verify the whole path end-to-end on 5,000 samples
(~1 minute). Its metrics are not valid — it only proves the pipeline runs.

In [ ]:
# Smoke test first: proves ingestion -> features -> fitting works end to end.
!python train.py --quick-test

In [ ]:
# Full production run on the complete master dataset.
!python train.py

## 5. Visual Evaluation & Benchmark Comparison

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np

report_path = 'saved_models/test_evaluation_report.json'
if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        report = json.load(f)
    
    models = ['Subword NLP', 'PhishNet Deep NN', 'Tabular LightGBM', 'Hybrid LightGBM', 'MoE Champion', 'Serving Fusion']
    keys = ['baseline_nlp', 'phishnet_deep_nn', 'tabular_lgbm', 'hybrid_lgbm', 'champion_moe', 'serving_fusion']
    
    accuracies = [report[k]['accuracy'] * 100 for k in keys]
    f1_scores = [report[k]['f1_score'] * 100 for k in keys]
    roc_aucs = [report[k]['roc_auc'] * 100 for k in keys]
    
    x = np.arange(len(models))
    width = 0.25
    
    plt.figure(figsize=(12, 6))
    plt.bar(x - width, accuracies, width, label='Accuracy (%)', color='#3b82f6')
    plt.bar(x, f1_scores, width, label='F1-Score (%)', color='#10b981')
    plt.bar(x + width, roc_aucs, width, label='ROC-AUC (%)', color='#8b5cf6')
    
    plt.xlabel('Model Architecture', fontweight='bold', fontsize=12)
    plt.ylabel('Score (%)', fontweight='bold', fontsize=12)
    plt.title('🛡️ PhishGuard 2.0 Held-Out Benchmark Performance (180,000+ Test Samples)', fontweight='bold', fontsize=14)
    plt.xticks(x, models, rotation=15, fontweight='bold')
    plt.ylim([85, 100])
    plt.legend(loc='lower right')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
    
    print("\n" + "=" * 70)
    print("🏆 CHAMPION MIXTURE-OF-EXPERTS (MoE) TEST BENCHMARK:")
    print("=" * 70)
    for k, v in report['champion_moe'].items():
        print(f"  • {k.upper():<15}: {v}")
    print("=" * 70)
else:
    print("No report found yet. Run Cell 3 first!")

## 6. Download Model Artifacts (`saved_models.zip`)

In [ ]:
# Download the packaged model bundle to your PC
import os
from google.colab import files

zip_file = 'saved_models.zip'
if os.path.exists(zip_file):
    size_mb = os.path.getsize(zip_file) / (1024 * 1024)
    print(f"📦 Ready to download '{zip_file}' ({size_mb:.2f} MB)...")
    files.download(zip_file)
else:
    print(f"⚠️ '{zip_file}' not found. Please ensure train.py has completed.")